In [25]:
from datetime import date, timedelta

from data_engineering.flights.flight_type import get_flights
import polars as pl
end_date = date(2025,5, 20)
start_date = end_date - timedelta(days=2)
df = get_flights(start_date,end_date)

In [34]:
df.filter(pl.col("landing_airport_ident") == "KHSA") # how is this possibly 0? 

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a2f35e""","""N29CL""","""N29CL""",2025-05-18 14:52:47.230,"""AZU""",2025-05-18 15:47:58.240,"""KHSA""",false,false,false,false,"""WW24""","""JORDAN AIR LLC""","""IAI 1124 Westwind""","""A1"""
"""a8e366""","""CAP2272""","""N672CP""",2025-05-18 14:37:53.360,"""US-6205""",2025-05-18 14:52:49.600,"""KHSA""",false,false,false,false,"""C182""","""CIVIL AIR PATROL""",null,"""A1"""
"""a96c55""","""N706RF""","""N706RF""",2025-05-18 19:45:58.120,"""KLFT""",2025-05-18 20:36:39.610,"""KHSA""",false,false,false,false,"""S22T""","""DEEPER SHADES LLC""","""CIRRUS SR-22T Turbo""","""A1"""
"""a9d4cc""","""N732ZV""","""N732ZV""",2025-05-18 23:03:27.460,"""KJKA""",2025-05-18 23:47:32.030,"""KHSA""",false,false,false,false,"""C210""","""OPPENHEIM DANIEL A""","""CESSNA 201 Centurion""","""A1"""
"""ad8a4b""","""N9714L""","""N9714L""",2025-05-18 19:56:35.260,"""US-6205""",2025-05-18 20:50:57.280,"""KHSA""",false,false,false,false,"""BE19""","""WEBAIRE LLC""","""BEECH 19 Sport""","""A1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ae1ed8""","""SH083""","""166083""",2025-05-19 20:20:00.960,"""KNDZ""",2025-05-19 21:48:53.810,"""KHSA""",false,false,true,false,"""TEX2""",null,"""Raytheon T-6B Texan II""","""A1"""
"""ae4a28""","""BB159""","""166159""",2025-05-19 23:48:15.070,"""KNDZ""",2025-05-20 00:37:10.240,"""KHSA""",false,false,true,false,"""TEX2""",null,"""Raytheon T-6B Texan II""","""A1"""
"""ae6274""","""ROKT11""","""163624""",2025-05-19 12:40:20.550,"""KNPA""",2025-05-19 13:48:41.860,"""KHSA""",false,false,true,false,"""HAWK""",null,null,"""A6"""


In [35]:
last_flight_per_icao = (
    df
    .filter(pl.col("ladd"))
    .sort(["icao", "landing_time"], descending=[False, True])
    .unique(subset=["icao"], keep="first")
)

last_landed_at_kteb = (
    last_flight_per_icao
    .filter(pl.col("landing_airport_ident") == "KHSA")
    .select([
        "icao",
        "callsign",
        "registration",
        "takeoff_time",
        "takeoff_airport_ident",
        "landing_time",
        "landing_airport_ident",
        "aircraft_type",
        "category",
    ])
    .sort("landing_time", descending=True)
)

last_landed_at_kteb

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,aircraft_type,category
str,str,str,datetime[ms],str,datetime[ms],str,str,str


In [6]:
from pia.ladd_aircraft_pia_candidates import get_ladd_pia_candidates


start_date = date(2025,5,20)
end_date = date(2025,5,21)
canidate_start_date = max(date(2024,1,1), start_date - timedelta(days=365))
df_flights = get_flights(canidate_start_date, end_date)
df_flights = df_flights.filter(pl.col("ladd") | pl.col("pia"))
ladd_pia_candidates = get_ladd_pia_candidates(df_flights)

In [8]:
last_landed_at_kteb.filter(pl.col("icao").is_in(ladd_pia_candidates))

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,aircraft_type,category
str,str,str,datetime[ms],str,datetime[ms],str,str,str
"""aa1b0a""","""N750LG""","""N750LG""",2025-05-19 23:29:19.430,"""KVNY""",2025-05-20 03:55:58.680,"""KTEB""","""GLF5""","""A3"""
"""adaa50""","""N98RP""","""N98RP""",2025-05-19 21:45:03.290,"""TJCP""",2025-05-20 02:24:56.890,"""KTEB""","""F2TH""","""A2"""
"""a19149""","""TWY330""","""N20SQ""",2025-05-19 23:56:35.120,"""KFOK""",2025-05-20 00:25:25.630,"""KTEB""","""GLEX""","""A3"""
"""ab48aa""","""WWI82""","""N826RP""",2025-05-19 22:15:05.320,"""KFOK""",2025-05-19 22:42:38.880,"""KTEB""","""GLF4""","""A2"""
"""a7d464""","""N603WM""","""N603WM""",2025-05-19 20:08:30.890,"""KLEB""",2025-05-19 20:49:50.780,"""KTEB""","""E55P""","""A2"""
…,…,…,…,…,…,…,…,…
"""a06f49""","""N127SF""","""N127SF""",2025-05-18 17:47:33.260,"""KBCT""",2025-05-18 20:20:11.890,"""KTEB""","""G280""","""A2"""
"""a11c37""","""N1705C""","""N1705C""",2025-05-18 14:44:40.210,"""MGPP""",2025-05-18 19:19:11.990,"""KTEB""","""C68A""","""A2"""
"""aaab60""","""N787JS""","""N787JS""",2025-05-18 17:15:01.670,"""KBNA""",2025-05-18 19:06:19.960,"""KTEB""","""GLF4""","""A2"""


# 2

In [38]:

from datetime import date, datetime, time, timedelta
from pathlib import Path
import polars as pl

from data_engineering.flights.flight_type import flights_df_to_flights, get_flights
from pia.ladd_aircraft_pia_candidates import get_ladd_pia_candidates
from pia.resolver import get_canidate_icaos, resolve_pia_mappings

def keep_start_window_with_prior_anchor(df_flights: pl.DataFrame, start_date: date) -> pl.DataFrame:
    start_datetime = datetime.combine(start_date, time.min)
    flight_key = ["icao", "takeoff_time"]

    flights_in_window = df_flights.filter(pl.col("takeoff_time") >= start_datetime)
    prior_anchor_flights = (
        df_flights
        .filter(
            (pl.col("ladd"))
            & (~pl.col("pia"))
            & (pl.col("takeoff_time") <= start_datetime)
        )
        .sort(["icao", "takeoff_time"], descending=[False, True])
        .unique(subset=["icao"], keep="first")
    )

    return (
        pl.concat([prior_anchor_flights, flights_in_window])
        .unique(subset=flight_key, keep="first")
        .sort(flight_key)
    )

canidate_start_date = max(date(2024,1,1), start_date - timedelta(days=365))
df_flights = get_flights(canidate_start_date, end_date)
df_flights = df_flights.filter(pl.col("ladd") | pl.col("pia"))
ladd_pia_candidates = get_ladd_pia_candidates(df_flights)
print(f"{len(ladd_pia_candidates)} ladd pia candidates")
df_flights = df_flights.filter(pl.col("icao").is_in(ladd_pia_candidates) | pl.col("pia"))
df_flights = keep_start_window_with_prior_anchor(df_flights, start_date)
df = df_flights.clone()
df_flights_2 = get_flights(end_date, end_date + timedelta(days=10)) # We proboably don't need most of this just pree ladd flight is what's required.
df_flights_2 = df_flights_2.filter(pl.col("icao").is_in(ladd_pia_candidates) | pl.col("pia"))
df_flights = pl.concat([df_flights, df_flights_2]) # We still have pia source nodes present. 
flights = flights_df_to_flights(df_flights)

3526 ladd pia candidates


# something

In [44]:
df = df.sort(["icao", "takeoff_time"], descending=[False, True]).unique(subset=["icao"], keep="first")

In [45]:
df.filter(pl.col("landing_airport_ident") == "KHSA")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str


#

In [37]:
df_flights.sort(["icao", "takeoff_time"], descending=[False, True])

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""0b425c""","""TGEZF""","""TG-EZF""",2025-05-18 13:27:25.100,"""KBIF""",2025-05-18 16:12:46.720,"""MX-0222""",false,true,false,false,"""C25C""",null,"""CESSNA 525C Citation CJ4""","""A2"""
"""0c4146""","""HI985""","""HI985""",2025-05-30 04:19:29.390,"""MWCR""",2025-05-30 05:01:58.620,"""MKKJ""",false,true,false,false,"""C680""",null,"""CESSNA 680 Citation Sovereign""","""A2"""
"""0c4146""","""HI985""","""HI985""",2025-05-30 00:54:37.870,"""MBPV""",2025-05-30 02:17:22.120,"""MWCR""",false,true,false,false,"""C680""",null,"""CESSNA 680 Citation Sovereign""","""A2"""
"""0c4146""","""HI985""","""HI985""",2025-05-29 23:01:25.640,"""MDMC""",2025-05-29 23:21:53.780,"""MBPV""",false,true,false,false,"""C680""",null,"""CESSNA 680 Citation Sovereign""","""A2"""
"""0c4146""","""HI985""","""HI985""",2025-05-25 18:10:11.490,"""TJSJ""",2025-05-25 18:17:51.430,"""TJAB""",false,true,false,false,"""C680""",null,"""CESSNA 680 Citation Sovereign""","""A2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""e09491""","""LVIRQ""","""LV-IRQ""",2025-05-26 08:27:51.810,"""LEGE""",2025-05-26 09:12:37.290,"""LFMN""",false,true,false,false,"""GLF5""",null,"""GULFSTREAM 5""","""A3"""
"""e09491""","""LVIRQ""","""LV-IRQ""",2025-05-22 13:07:52.680,"""LFMN""",2025-05-22 13:47:38.210,"""ES-0127""",false,true,false,false,"""GLF5""",null,"""GULFSTREAM 5""","""A3"""
"""e09491""","""LVIRQ""","""LV-IRQ""",2025-05-21 23:34:41.900,"""MX-2029""",2025-05-22 10:42:52.650,"""LFMN""",false,true,false,false,"""GLF5""",null,"""GULFSTREAM 5""","""A3"""


In [20]:
# length of flights should be 1 day (100k) and a bit more
len(flights)

29236

In [22]:
from pia.resolver import build_graph, remove_pia_source_nodes
g = build_graph(flights)

In [23]:
remove_pia_source_nodes(g)

73 pia source nodes remove
a039a2 2025-05-20 13:27:47.480000 KHSA
a03969 2025-05-20 14:34:37.510000 57D
a4fe94 2025-05-20 14:55:41.700000 04CA
a03b9b 2025-05-20 17:31:08.040000 4U8
a0fe6e 2025-05-20 17:39:58 KRST
a0feb4 2025-05-20 18:39:10.640000 KX13
a0feb4 2025-05-20 19:20:00.590000 GA25
a03b35 2025-05-20 20:16:52.600000 7TX7
a0fe56 2025-05-21 04:06:11.440000 KAKQ
a02008 2025-05-21 07:27:08.290000 LTAZ
a02008 2025-05-21 10:02:01.080000 TR-0004
a100b1 2025-05-21 17:35:27.020000 US-7414
a0fe56 2025-05-22 18:46:56.380000 K03B
a0feb4 2025-05-22 20:42:58.170000 KAAF
a0fe56 2025-05-22 23:00:52.840000 OK49
a0fe56 2025-05-23 00:25:39.660000 K3E0
a0ff0e 2025-05-23 01:28:47.320000 3MD8
a038c0 2025-05-23 01:40:05.170000 4IL8
a4f13f 2025-05-23 02:37:33.290000 XS12
a0ff0e 2025-05-23 02:47:44.870000 VG36
a02008 2025-05-23 12:51:47.810000 TR-0014
a0feb4 2025-05-23 14:32:09.140000 75FL
a0fe56 2025-05-23 14:55:22.880000 NM89
a0ff0e 2025-05-23 15:25:16.860000 6NR6
a0feb4 2025-05-23 15:50:33.730000 KMJ

Exception: pia source nodes present

In [46]:
from airports.airportLookup import AirportLookup

airport_lookup = AirportLookup()
khsa_coords = airport_lookup.get_airport_coordinates("KHSA")
if khsa_coords is None:
    raise ValueError("KHSA not found in airport lookup")

khsa_lat, khsa_lon = khsa_coords
nearby_airports = airport_lookup.getAirportsWithinRadius(khsa_lat, khsa_lon, radius_km=100)
nearby_airport_idents = [airport.ident for airport in nearby_airports]

flights_within_100km_of_khsa = (
    df
    .filter(pl.col("landing_airport_ident").is_in(nearby_airport_idents))
    .sort("takeoff_time")
)

flights_within_100km_of_khsa

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""abce74""","""N86WB""","""N86WB""",2024-12-19 16:34:08.110,"""82LA""",2024-12-19 16:41:36.860,"""KNEW""",false,true,false,false,"""SR22""","""HARRIS MICHAEL G""","""CIRRUS SR-22""","""A1"""
"""a8b5cb""","""N660SB""","""N660SB""",2025-04-27 14:52:16.750,"""KGAO""",2025-04-27 15:18:09.770,"""32LS""",false,true,false,false,"""R66""","""SOUTHERN EXPRESS LLC""","""ROBINSON R-66""","""A7"""
"""a13f8c""","""N18NN""","""N18NN""",2025-05-18 17:03:33.350,"""KSFO""",2025-05-18 20:38:23.890,"""KNEW""",false,true,false,false,"""GLEX""","""SLJ ACQUISITION LLC""","""BOMBARDIER BD-700 Global 6000/…","""A3"""
"""acd11a""","""N925DT""","""N925DT""",2025-05-18 17:09:23.720,"""KSUS""",2025-05-18 18:26:02.700,"""5MS5""",false,true,false,false,"""E55P""","""925DT LLC""","""EMBRAER EMB-505 Phenom 300""","""A2"""
"""ad8471""","""N970JD""","""N970JD""",2025-05-18 21:19:14.560,"""KF70""",2025-05-19 00:10:57.410,"""KMSY""",false,true,false,false,"""F900""","""VOICE OF THE COVENANT CHURCH I…","""DASSAULT Falcon 900""","""A2"""


In [47]:
df_flights = get_flights(date(2025,5,1), date(2025,5,2))
df_flights = df_flights.filter(pl.col("ladd") | pl.col("pia"))
df_flights

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""06a05a""","""A7AAH""","""A7-AAH""",2025-05-01 12:16:33.570,"""EGSS""",2025-05-01 18:42:35.060,"""OTHH""",false,true,false,false,"""A343""",null,"""AIRBUS A-340-300""","""A5"""
"""06a0a4""","""A7MBK""","""A7-MBK""",2025-05-01 22:24:42.490,"""EGGW""",2025-05-02 02:16:56.480,"""LTAR""",false,true,false,false,"""A320""",null,"""AIRBUS A-320""","""A3"""
"""0a8013""","""C6WAV""","""C6-WAV""",2025-05-01 10:46:40.280,"""MYBS""",2025-05-01 11:16:02.440,"""KFLL""",false,true,false,false,"""C208""",null,"""CESSNA 208 Caravan""","""A1"""
"""0a8013""","""C6WAV""","""C6-WAV""",2025-05-01 12:18:52.970,"""KFLL""",2025-05-01 12:31:17.960,"""KOPF""",false,true,false,false,"""C208""",null,"""CESSNA 208 Caravan""","""A1"""
"""0a8013""","""C6WAV""","""C6-WAV""",2025-05-01 13:29:28.180,"""KOPF""",2025-05-01 13:35:42.230,"""KFLL""",false,true,false,false,"""C208""",null,"""CESSNA 208 Caravan""","""A1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""e49f46""","""PPNGM""","""PP-NGM""",2025-05-01 12:06:40.310,"""SBGR""",2025-05-01 13:27:24.300,"""SSAU""",false,true,false,false,"""GLF5""",null,"""GULFSTREAM 5""","""A3"""
"""e49f94""","""PSFST""","""PS-FST""",2025-05-01 02:49:36.250,"""SBGR""",2025-05-01 03:17:51.240,"""SIMK""",false,true,false,false,"""FA7X""",null,"""DASSAULT Falcon 7X""","""A2"""
"""e49f94""","""PSFST""","""PS-FST""",2025-05-01 07:51:32.580,"""TGPY""",2025-05-01 11:06:40.900,"""KFLL""",false,true,false,false,"""FA7X""",null,"""DASSAULT Falcon 7X""","""A2"""


In [1]:
from pia.main import run_pia_algorithm
from datetime import date
import polars as pl
from data_engineering.flights.flight_type import get_flights
start_date = date(2025,5,20)
end_date = date(2025,5,21)
mappings, g = run_pia_algorithm(start_date, end_date)

3523 ladd pia candidates
Number of flights: 6367
Number of subgraphs: 4021
Number of nodes: 152
Number of nodes: 31
Number of nodes: 356
Number of nodes: 44
Number of nodes: 6
Number of nodes: 15
Number of nodes: 9
Number of nodes: 4
Number of nodes: 13
Number of nodes: 3
Number of nodes: 6
Number of nodes: 12
Number of nodes: 8
Number of nodes: 6
Number of nodes: 6
Number of nodes: 1
Number of nodes: 1
Number of nodes: 1
{'a1031b_2025-05-20_12-19': 'aa0602', 'a1010f_2025-05-20_18-52': 'a132b3', 'a0fef4_2025-05-20_01-24': 'acc38b', 'a03c16_2025-05-20_03-16': 'a660f6', 'a03c16_2025-05-20_00-10': 'a660f6', 'a1010f_2025-05-20_17-39': 'a132b3', 'a1031b_2025-05-20_20-04': 'c00d53', 'a1010f_2025-05-20_23-06': 'adc192', 'a0feab_2025-05-20_23-44': 'a63e21', 'a03b3f_2025-05-20_06-35': 'a83c5f', 'a03b8a_2025-05-20_14-14': 'a83c5f', 'a039a2_2025-05-20_16-16': 'a7a5b6', 'a03b35_2025-05-20_20-16': '0c4155', 'a0398c_2025-05-20_12-06': 'a02c8a', 'a0398c_2025-05-19_23-31': 'a02c8a', 'a03b1d_2025-05-20

In [3]:
# check the mappings for where a039a2 flights map to
for key in mappings.keys():
    if key.startswith("a039a2"):
        print(key, mappings[key])

a039a2_2025-05-20_16-16 a7a5b6
a039a2_2025-05-20_13-27 abf0ca


In [6]:
from data_engineering.flights.flight_type import get_flights
df_flights = get_flights(date(2025,1,1), end_date)

In [9]:
import polars as pl
df_flights.filter(pl.col("icao") == "a7a5b6").sort("landing_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a7a5b6""","""CXK691""","""N592K""",2025-01-04 15:34:09.170,"""20FL""",2025-01-04 15:57:21.460,"""FL86""",false,true,false,false,"""P28A""","""ATP AIRCRAFT 3 LLC""","""PIPER PA-28-140/150/160/180""","""A1"""
"""a7a5b6""","""CXK691""","""N592K""",2025-01-05 15:21:34.380,"""KCHN""",2025-01-05 15:58:56.010,"""94FL""",false,true,false,false,"""P28A""","""ATP AIRCRAFT 3 LLC""","""PIPER PA-28-140/150/160/180""","""A1"""
"""a7a5b6""","""CXK691""","""N592K""",2025-01-06 14:27:21.610,"""KFMY""",2025-01-06 14:40:15.270,"""KFMY""",false,true,false,false,"""P28A""","""ATP AIRCRAFT 3 LLC""","""PIPER PA-28-140/150/160/180""","""A1"""
"""a7a5b6""","""CXK691""","""N592K""",2025-01-07 12:51:57.950,"""US-5120""",2025-01-07 13:01:47.110,"""KSRQ""",false,true,false,false,"""P28A""","""ATP AIRCRAFT 3 LLC""","""PIPER PA-28-140/150/160/180""","""A1"""
"""a7a5b6""","""CXK691""","""N592K""",2025-01-07 13:12:36.950,"""KSRQ""",2025-01-07 13:50:02.920,"""KFMY""",false,true,false,false,"""P28A""","""ATP AIRCRAFT 3 LLC""","""PIPER PA-28-140/150/160/180""","""A1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a7a5b6""","""CXK691""","""N592K""",2025-05-15 21:22:17.180,"""KFPR""",2025-05-15 21:28:32.800,"""FD57""",false,true,false,false,"""P28A""","""ATP AIRCRAFT 3 LLC""","""PIPER PA-28-140/150/160/180""","""A1"""
"""a7a5b6""","""CXK691""","""N592K""",2025-05-15 21:31:39.030,"""FD57""",2025-05-15 21:57:07.120,"""KSUA""",false,true,false,false,"""P28A""","""ATP AIRCRAFT 3 LLC""","""PIPER PA-28-140/150/160/180""","""A1"""
"""a7a5b6""","""CXK691""","""N592K""",2025-05-18 21:30:26.750,"""KSUA""",2025-05-18 21:35:36.140,"""KSUA""",false,true,false,false,"""P28A""","""ATP AIRCRAFT 3 LLC""","""PIPER PA-28-140/150/160/180""","""A1"""


In [10]:
import polars as pl
df_flights.filter(pl.col("icao") == "a039a2").sort("landing_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a039a2""","""DCM2493""",null,2025-02-18 19:32:37.940,"""US-6205""",2025-02-18 19:59:16.170,"""KHSA""",true,false,false,false,null,null,null,"""A2"""
"""a039a2""","""DCM2493""",null,2025-03-03 15:29:43.200,"""KHSA""",2025-03-03 17:21:14.820,"""KICT""",true,false,false,false,null,null,null,"""A2"""
"""a039a2""","""DCM2493""",null,2025-03-03 17:54:22.100,"""KICT""",2025-03-03 20:12:51.600,"""KBZN""",true,false,false,false,null,null,null,"""A2"""
"""a039a2""","""DCM2493""",null,2025-03-03 21:00:04.190,"""KBZN""",2025-03-03 22:25:15.870,"""KAPA""",true,false,false,false,null,null,null,"""A2"""
"""a039a2""","""DCM2493""",null,2025-03-09 13:07:04.010,"""KAPA""",2025-03-09 14:36:33.940,"""05MT""",true,false,false,false,null,null,null,"""A2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a039a2""","""DCM2493""",null,2025-05-14 13:16:59.420,"""KJWN""",2025-05-14 14:52:14.460,"""KOKC""",true,false,false,false,null,null,null,"""A2"""
"""a039a2""","""DCM2493""",null,2025-05-14 21:46:34.580,"""KOKC""",2025-05-14 22:56:23.540,"""KBTR""",true,false,false,false,null,null,null,"""A2"""
"""a039a2""","""DCM2493""",null,2025-05-14 23:29:19.780,"""KBTR""",2025-05-14 23:49:09.970,"""KHSA""",true,false,false,false,null,null,null,"""A2"""
